# Calibration 04-2 — L-BFGS-B Fit (2019-2020, 7 age groups)

**설정** (NM 과 동일):
- 22D 파라미터 + Gaussian seasonality + first_peak_only + auto-seed + immunity=0.3
- L-BFGS-B: gradient-based (finite difference). 빠르지만 local minimum 가능.
- `max_iterations=500` (NM 대비 ¼)

**예상 시간**: 10-30분.

**주의**: 좁힌 bounds + min_rate=0.1 적용 후 L-BFGS-B 가 corner solution 으로 도망가는 문제 해결됐는지 확인할 시즌. 이전 fit (cosine_archive) 의 β=0.001 corner 와 비교.

출력: `outputs/calibration/2019-2020_by_age_LBFGS.json` + 시각화 PNG.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from kt_data import ILI_AGE_GROUPS
from kt_epimodel.calibration.ili_target import (
    load_ili_target_by_age, simulation_to_ili_by_age,
)
from kt_epimodel.calibration.optimizer import (
    optimize_calibration_by_age, save_result,
)
from kt_epimodel.calibration.simple_model import (
    build_aggregated_inputs, simulate_aggregated,
    estimate_initial_infected_from_ili,
)
from kt_epimodel.model.compartments import IDX_S
from kt_epimodel.model.parameters import (
    DiseaseParameters, ModelParameters,
)

OUT = Path('../outputs/calibration'); OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. ILI target 확인

In [ ]:
target_age = load_ili_target_by_age('2019-2020', first_peak_only=True, first_peak_end_week=26)
weeks = target_age['week_in_season']

## 2. L-BFGS-B fit (max_iter=500)

Gradient-based — finite difference로 ∂NLL/∂param 추정 후 quasi-Newton step.
Bounds 적용 (scipy 1.7+). 빠르나 local minimum 도달 가능.

In [ ]:
t0 = time.time()
result_lbfgs = optimize_calibration_by_age(
    season='2019-2020',
    use_data_seed=True,
    gamma_report_assumed=2.0,
    seed_e_factor=0.5,
    initial_immunity=0.3,
    method='L-BFGS-B',
    max_iterations=500,
    first_peak_only=True,
    first_peak_end_week=26,
    t_span=(0.0, 364.0),
    verbose=True,
)
print(f"\nTotal elapsed: {(time.time() - t0) / 60:.1f} min")
save_result(result_lbfgs, OUT / '2019-2020_by_age_LBFGS.json')

## 3. Fit 결과 — 7 그룹 관측 vs 예측

In [ ]:
inputs = build_aggregated_inputs()
pop_15 = inputs['pop_15'].flatten()

disease = DiseaseParameters(
    seasonality_mode=result_lbfgs.seasonality_mode,
    seasonality_amp=result_lbfgs.seasonality_amp,
    seasonality_base=result_lbfgs.seasonality_base,
    seasonality_sigma=result_lbfgs.seasonality_sigma,
    seasonality_peak_day=result_lbfgs.seasonality_peak_day,
)
params = ModelParameters(disease=disease).with_calibration(result_lbfgs.calibration)

if result_lbfgs.use_data_seed:
    seed_by_age = estimate_initial_infected_from_ili(
        '2019-2020', pop_15,
        gamma_report_assumed=result_lbfgs.gamma_report_assumed,
    )
else:
    seed_by_age = None

sim = simulate_aggregated(
    params, inputs,
    seed_total=result_lbfgs.seed_total if not result_lbfgs.use_data_seed else 0.0,
    seed_by_age=seed_by_age,
    initial_immunity=result_lbfgs.initial_immunity,
    t_span=(0.0, 364.0),
)
S_age = sim.states[:, IDX_S, :, :].sum(axis=-1)
daily_inc = -np.diff(S_age, axis=0)
predictions = simulation_to_ili_by_age(
    daily_inc, pop_15, result_lbfgs.calibration.gamma_report, n_weeks=52,
)

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)
for i, ag in enumerate(ILI_AGE_GROUPS):
    ax = axes[i // 4, i % 4]
    ax.plot(weeks, target_age['ili_rates'][ag], 'ko-', markersize=3, label='observed')
    ax.plot(weeks, predictions[ag], 'b-', linewidth=2, label='predicted (L-BFGS-B)')
    ax.axvspan(26, 52, alpha=0.1, color='gray')
    ax.set_title(f'Age {ag}')
    ax.grid(True, alpha=0.3)
    if i % 4 == 0:
        ax.set_ylabel('ILI / 1000')
    if i // 4 == 1:
        ax.set_xlabel('Week')
axes[1, 3].axis('off')
axes[0, 0].legend(loc='upper right', fontsize=8)
fig.suptitle(
    f'2019-2020 L-BFGS-B fit (mode={result_lbfgs.seasonality_mode}) — '
    f'NLL {result_lbfgs.nll_initial:.0f} → {result_lbfgs.nll:.0f}',
    fontsize=13,
)
plt.tight_layout()
plt.savefig(OUT / '2019-2020_by_age_LBFGS_fit.png', dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
AGES = ['0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39',
        '40-44','45-49','50-54','55-59','60-64','65-69','70+']
axes[0].bar(np.arange(15), result_lbfgs.calibration.phi, color='C0')
axes[0].axhline(1.0, color='red', linestyle='--', label='reference')
axes[0].set_xticks(np.arange(15)); axes[0].set_xticklabels(AGES, rotation=45)
axes[0].set_ylabel(r'$\phi_a$')
axes[0].set_title('Age-specific susceptibility (L-BFGS-B fit)')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

channels = ['home', 'work', 'school', 'other']
betas = [result_lbfgs.calibration.beta_h, result_lbfgs.calibration.beta_w,
         result_lbfgs.calibration.beta_s, result_lbfgs.calibration.beta_o]
axes[1].bar(channels, betas, color=['C0', 'C1', 'C2', 'C3'])
axes[1].set_ylabel(r'$\beta_{ch}$')
axes[1].set_title(
    f'Channel β (γ_r={result_lbfgs.calibration.gamma_report:.3f}, '
    f'amp={result_lbfgs.seasonality_amp:.2f}, base={result_lbfgs.seasonality_base:.2f}, '
    f'σ={result_lbfgs.seasonality_sigma:.0f})'
)
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUT / '2019-2020_by_age_LBFGS_phi_beta.png', dpi=120)
plt.show()

print(pl.DataFrame([{
    'method': result_lbfgs.method,
    'nll_initial': result_lbfgs.nll_initial,
    'nll_final': result_lbfgs.nll,
    'n_evals': result_lbfgs.n_evaluations,
    'elapsed_min': result_lbfgs.elapsed_seconds / 60,
    'beta_h': result_lbfgs.calibration.beta_h,
    'beta_w': result_lbfgs.calibration.beta_w,
    'beta_s': result_lbfgs.calibration.beta_s,
    'beta_o': result_lbfgs.calibration.beta_o,
    'gamma_report': result_lbfgs.calibration.gamma_report,
    'amp': result_lbfgs.seasonality_amp,
    'base': result_lbfgs.seasonality_base,
    'sigma': result_lbfgs.seasonality_sigma,
    'phi_min': float(result_lbfgs.calibration.phi.min()),
    'phi_max': float(result_lbfgs.calibration.phi.max()),
}]))